# DS 227 &middot; Knowledge Discovery in Data &mdash; Week 5 Lab
## Missing Values &amp; Outliers

Real data has holes and surprises. This week you find **missing values** and **outliers**,
and &mdash; the hard part &mdash; decide on purpose what to do about each.

**How long:** about 45 minutes. Nothing to install.

Work top to bottom. The Stretch section at the end is optional.

---
## Part 0 &middot; Find the missing values

`NaN` marks a hole. `isna().sum()` counts them per column &mdash; the first thing to know
about any dataset. Run the cell.

In [ ]:
import pandas as pd, numpy as np
df = pd.DataFrame({
    "city":   ["Cebu", "Davao", "Iloilo", "Baguio", "Tacloban"],
    "temp":   [31, np.nan, 33, 29, np.nan],
    "rain_mm":[12, 4, np.nan, 8, 0],
})
print(df.isna().sum())
print("\nrows with any gap:\n", df[df.isna().any(axis=1)])

**Answer here** (double-click to edit):

1. Which column has the most missing values, and how many rows are affected in total?
   &rarr; *your answer*

2. Why is counting gaps *per column* more useful than a single total when deciding what to
   do next?
   &rarr; *your answer*

---
## Part 1 &middot; Drop or fill &mdash; a decision

You can **drop** rows with gaps or **fill** them. Neither is free; each changes what the
data says. Run the cell.

In [ ]:
import pandas as pd, numpy as np
df = pd.DataFrame({"temp": [31, np.nan, 33, 29, np.nan]})

print("drop:\n", df.dropna(), "\n")
print("fill with mean:\n", df.fillna(df["temp"].mean()))

**Answer here:**

1. Dropping left 3 rows; filling kept 5. What did each choice cost &mdash; what information
   did you lose or invent?
   &rarr; *your answer*

2. Filling with the mean pulls those rows toward the average. When would that be
   misleading, and what is an alternative fill?
   &rarr; *your answer*

---
## Part 2 &middot; Spot outliers with the IQR rule

An **outlier** sits far from the rest. The IQR rule flags points below Q1 &minus; 1.5&times;IQR
or above Q3 + 1.5&times;IQR. Run the cell.

In [ ]:
import pandas as pd
s = pd.Series([12, 14, 13, 15, 11, 14, 98, 13, 12])   # 98 looks off

q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"bounds: {low:.1f} to {high:.1f}")
print("outliers:", s[(s < low) | (s > high)].tolist())

**Answer here:**

1. Which value was flagged, and roughly how far outside the bounds was it?
   &rarr; *your answer*

2. The IQR rule flagged `98` as *statistically* unusual. That is not the same as *wrong* &mdash;
   name one case where a real value could legitimately look like an outlier.
   &rarr; *your answer*

---
## Part 3 &middot; Decide, don't just delete

Flagging is easy; deciding is the work. Keep, cap, or remove &mdash; on purpose, and
write down why. Run the cell.

In [ ]:
import pandas as pd
s = pd.Series([12, 14, 13, 15, 11, 14, 98, 13, 12])

capped = s.clip(upper=s.quantile(0.75) + 1.5 * (s.quantile(0.75) - s.quantile(0.25)))
print("original mean:", round(s.mean(), 2))
print("capped mean:  ", round(capped.mean(), 2))

**Answer here:**

1. Capping barely changed the data but moved the mean a lot. What does that reveal about
   how sensitive the mean is to a single extreme value?
   &rarr; *your answer*

2. For *this* series, would you keep, cap, or drop the `98`? State your choice and the
   reason a reader would accept.
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Median vs mean

Compare `s.mean()` and `s.median()` for the series with the `98` in it. Which better
represents a 'typical' value here, and why is the median more robust to outliers?

In [ ]:
# your code here

### Stretch 2 &middot; Fill by group

Given a frame with a `region` and a `temp` column that has gaps, fill each missing
`temp` with that region's mean using `groupby(...).transform(...)`. Why is that better than
one global mean?

In [ ]:
# your code here

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds227", 5

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds227/lab/5/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 5 submission page](https://portal.latarak.com/course/ds227/lab/5/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.